# ChatUniTest Colab Runner (Clean)

Run cells in order: setup -> login -> start server -> create tunnel -> test request.

Optional: set `LORA_MODEL` to your own adapter (for example `your-username/my-testgen-lora`).

In [ ]:
!nvidia-smi

import os
import shutil

repo_dir = '/content/chatunitest-models'
venv_path = '/content/chatunitest_venv'

if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone https://github.com/alexli-77/chatunitest-models.git -b main {repo_dir}

if os.path.exists(venv_path):
    shutil.rmtree(venv_path)
!python3 -m venv {venv_path}
!{venv_path}/bin/python -m pip install --upgrade pip
!{venv_path}/bin/python -m pip install torch transformers peft flask bitsandbytes accelerate huggingface_hub requests

print('Setup complete.')

In [ ]:
import getpass
from huggingface_hub import login

print('Enter your Hugging Face access token (read for model download, write if you plan to push):')
token = getpass.getpass()
if token:
    login(token=token, add_to_git_credential=True)
    print('Hugging Face login successful.')
else:
    print('No token provided.')

In [ ]:
import os

repo_dir = '/content/chatunitest-models'
venv_path = '/content/chatunitest_venv'
server_script = f'{repo_dir}/model_server.py'

# Optional override for your fine-tuned adapter
# os.environ['LORA_MODEL'] = 'your-username/my-testgen-lora'

os.environ['HOST'] = '0.0.0.0'
os.environ['PORT'] = '1234'

!pkill -f model_server.py || true
!nohup {venv_path}/bin/python {server_script} > /content/server_log.txt 2>&1 &
!sleep 5
!tail -n 20 /content/server_log.txt || true

In [ ]:
import os
import re
import time

if not os.path.exists('/content/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

!pkill -f cloudflared || true
!rm -f /content/tunnel.log
get_ipython().system_raw('/content/cloudflared tunnel --url http://127.0.0.1:1234 > /content/tunnel.log 2>&1 &')

time.sleep(12)
log_content = open('/content/tunnel.log', 'r').read() if os.path.exists('/content/tunnel.log') else ''
urls = re.findall(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com', log_content)
public_url = urls[-1] if urls else None

if public_url:
    print(f'Public URL: {public_url}')
else:
    print('Could not find tunnel URL. Check /content/tunnel.log')

In [ ]:
import os
import re
import requests

health = requests.get('http://127.0.0.1:1234/health', timeout=20)
print('Local health:', health.status_code, health.text)

log_content = open('/content/tunnel.log', 'r').read() if os.path.exists('/content/tunnel.log') else ''
urls = re.findall(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com', log_content)
public_url = urls[-1] if urls else None

if public_url:
    payload = {
        'mode': 'COMPLETION',
        'projectPath': 'unknown',
        'assertionStyle': 'JUNIT',
        'staticSnapshot': 'public class A { int add(int a, int b) { return a + b; } }',
        'runtimeFacts': '',
        'max_tokens': 128,
        'temperature': 0.6
    }
    resp = requests.post(f'{public_url}/generation', json=payload, timeout=120)
    print('Remote status:', resp.status_code)
    print(resp.json())
else:
    print('No public URL available; run Cell 5 again.')